<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB03_taxon_directed_homologs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB03_taxon_directed_homologs.ipynb)

> **Opening this notebook from Canvas:** Select **File → Save a copy in Drive** before you begin. Work in your saved copy—not in the repository preview.

**Course release:** Fall 2026  
**Status:** Revised taxon-directed homolog-discovery workflow.

# NB03 — Taxon-directed homolog discovery

**Biological question:** How can we obtain a deliberately diverse, minimally redundant set of homologous protein sequences rather than simply accepting the highest-scoring BLAST hits from a single unrestricted search?

## Learning goals

By the end of this notebook, you should be able to:

1. Explain why taxon-directed sampling can improve biological diversity in a homolog set.
2. Use an NCBI Taxonomy ID (TaxID) to restrict a BLASTP search to a chosen taxonomic target.
3. Evaluate a candidate hit using E-value, percent identity, and query coverage.
4. Distinguish **homology discovery** from proof of **orthology**.
5. Save a reproducible, curated FASTA file and metadata for later multiple-sequence alignment and phylogenetic analysis.

**Inputs**

- `Data/00_project_sequence/translated_query_protein.faa`
- `Data/NB03_taxon_directed_homologs/taxon_targets.tsv`

**Required outputs**

- `homologs_raw.faa`
- `homologs_curated.faa`
- `homolog_metadata.tsv`
- `acquisition_parameters.tsv`
- `rejected_hits.tsv`

The notebook also creates two optional iTOL annotation files for later tree visualization.

> **Important interpretation:** The best BLAST hit within a taxon is a strong *candidate homolog*. It is not, by itself, proof that the sequence is the one-to-one ortholog of the query.

## 1. Connect Colab to Google Drive

Run this cell and authorize the Google account containing your course folder.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive is connected.")

ValueError: mount failed

## 2. Locate the course folders

The setup recognizes both supported course-folder locations automatically. Students normally use `MyDrive/BIOINFO4-5203-F26/`; the instructor copy may use `MyDrive/Teaching/BIOINFO4-5203-F26/`.

In [ ]:
from pathlib import Path

COURSE_FOLDER_NAME = "BIOINFO4-5203-F26"
NOTEBOOK_ID = "NB03_taxon_directed_homologs"

candidate_course_dirs = [
    Path("/content/drive/MyDrive") / COURSE_FOLDER_NAME,
    Path("/content/drive/MyDrive/Teaching") / COURSE_FOLDER_NAME,
]
existing_course_dirs = [path for path in candidate_course_dirs if path.exists()]

if existing_course_dirs:
    COURSE_DIR = existing_course_dirs[0]
else:
    COURSE_DIR = candidate_course_dirs[0]
    COURSE_DIR.mkdir(parents=True, exist_ok=True)
    print("Created the standard student course folder.")

QUERY_DIR = COURSE_DIR / "Data" / "00_project_sequence"
DATA_DIR = COURSE_DIR / "Data" / NOTEBOOK_ID
OUTPUT_DIR = COURSE_DIR / "Outputs" / NOTEBOOK_ID

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Course folder:", COURSE_DIR)
print("Query folder: ", QUERY_DIR)
print("Input folder: ", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

## 3. Install the Python packages used in this notebook

This workflow sends **remote BLASTP** jobs to NCBI. It therefore does not need a local copy of the `nr` database.

In [ ]:
%pip install -q biopython pandas

import pandas as pd
import Bio

print("pandas:", pd.__version__)
print("Biopython:", Bio.__version__)

## 4. Configuration and preflight check

This is the main configuration cell. Enter your real e-mail address when prompted; NCBI asks automated BLAST clients to identify themselves.

The default quality screen requires a candidate hit to cover at least **50% of the query sequence**. This is deliberately modest: it rejects obvious short fragments while retaining distant homolog candidates for later inspection.

In [ ]:
from pathlib import Path
from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW

# ---------- Input files ----------
QUERY_FASTA = QUERY_DIR / "translated_query_protein.faa"
TAXON_TARGETS = DATA_DIR / "taxon_targets.tsv"

# ---------- NCBI identification ----------
NCBI_EMAIL = input("Enter your e-mail address for NCBI requests: ").strip()
if "@" not in NCBI_EMAIL:
    raise ValueError("Please enter a valid e-mail address before continuing.")

Entrez.email = NCBI_EMAIL
Entrez.tool = "BIOINFO4-5203-F26_NB03"
NCBIWWW.email = NCBI_EMAIL
NCBIWWW.tool = "BIOINFO4-5203-F26_NB03"

# ---------- BLAST parameters ----------
BLAST_PROGRAM = "blastp"
BLAST_DATABASE = "nr"
EVALUE = 1e-5
HITLIST_SIZE = 10
MIN_QUERY_COVERAGE = 0.50

# NCBI asks automated BLAST clients not to contact the server
# more frequently than once every 10 seconds.
MIN_SECONDS_BETWEEN_BLAST_SUBMISSIONS = 10.0

# Retries are useful for temporary network or NCBI service errors.
MAX_RETRIES = 4
BACKOFF_START_SECONDS = 5.0

required = [QUERY_FASTA, TAXON_TARGETS]
missing = [p for p in required if not p.exists()]

print("\nExpected input files:")
for p in required:
    print(" ✓" if p.exists() else " ✗", p)

if missing:
    raise FileNotFoundError(
        "Missing required input file(s):\n" + "\n".join(str(p) for p in missing)
    )

## 5. Load and validate the query protein

The FASTA file should contain exactly one translated protein sequence.

In [ ]:
query_records = list(SeqIO.parse(QUERY_FASTA, "fasta"))

if len(query_records) != 1:
    raise ValueError(
        f"Expected exactly one protein sequence in {QUERY_FASTA.name}; "
        f"found {len(query_records)}."
    )

query_record = query_records[0]
query_seq = str(query_record.seq).replace(" ", "").replace("\n", "")
query_length = len(query_seq)

if query_length == 0:
    raise ValueError("The query sequence is empty.")

print("Query ID:    ", query_record.id)
print("Query length:", query_length, "amino acids")
print("Sequence:    ", query_seq[:60] + ("..." if query_length > 60 else ""))

## 6. Load and validate the TaxID target table

The table must contain two tab-delimited columns. The first column is a readable label; the second is an NCBI Taxonomy ID.

The notebook stops if the same TaxID appears more than once. Duplicate targets would waste remote BLAST searches and can unintentionally bias the final sequence set.

In [ ]:
targets = pd.read_csv(TAXON_TARGETS, sep="\t", dtype=str)

if targets.shape[1] < 2:
    raise ValueError("The taxon table must contain at least two tab-delimited columns.")

targets = targets.iloc[:, :2].copy()
targets.columns = ["label", "taxid"]
targets["label"] = targets["label"].str.strip()
targets["taxid"] = targets["taxid"].str.strip()

bad_taxids = targets[~targets["taxid"].str.fullmatch(r"\d+")]
if not bad_taxids.empty:
    display(bad_taxids)
    raise ValueError("One or more TaxIDs are not numeric.")

duplicate_taxids = targets[targets.duplicated("taxid", keep=False)]
if not duplicate_taxids.empty:
    print("Duplicate TaxIDs must be resolved before BLAST:")
    display(duplicate_taxids)
    raise ValueError("Duplicate TaxIDs found in taxon_targets.tsv.")

print(f"Loaded {len(targets)} unique taxonomic targets.")
display(targets.head(10))

### 6A. Taxonomy sanity check

Before spending time on BLAST searches, ask NCBI what organism or taxon each TaxID currently represents. This catches transcription errors and also helps distinguish species-level targets from broader taxonomic groups.

The readable labels in your table do **not** have to exactly match the NCBI scientific names; they may be short instructional labels.

In [ ]:
def fetch_taxonomy_records(taxids, batch_size=100):
    records = []
    taxids = list(map(str, taxids))
    for start in range(0, len(taxids), batch_size):
        batch = taxids[start:start + batch_size]
        with Entrez.efetch(
            db="taxonomy",
            id=",".join(batch),
            retmode="xml"
        ) as handle:
            records.extend(Entrez.read(handle))
    return records

taxonomy_records = fetch_taxonomy_records(targets["taxid"])
taxonomy_lookup = {
    str(rec["TaxId"]): {
        "ncbi_scientific_name": rec["ScientificName"],
        "rank": rec.get("Rank", "")
    }
    for rec in taxonomy_records
}

validated_targets = targets.copy()
validated_targets["ncbi_scientific_name"] = validated_targets["taxid"].map(
    lambda x: taxonomy_lookup.get(x, {}).get("ncbi_scientific_name", "NOT FOUND")
)
validated_targets["rank"] = validated_targets["taxid"].map(
    lambda x: taxonomy_lookup.get(x, {}).get("rank", "")
)

display(validated_targets)

not_found = validated_targets[
    validated_targets["ncbi_scientific_name"].eq("NOT FOUND")
]
if not not_found.empty:
    print("WARNING: Some TaxIDs were not returned by NCBI. Check these before continuing.")

## 7. BLAST and metadata helper functions

For each requested TaxID, the notebook:

1. runs BLASTP against NCBI `nr` restricted to that taxon;
2. examines the returned candidates in BLAST rank order;
3. chooses the first candidate meeting the minimum query-coverage criterion;
4. retrieves the full protein record from NCBI;
5. records sequence and metadata immediately so the notebook can resume after an interruption.

**Why no automatic rejection of “hypothetical” or “uncharacterized” proteins?** Those words describe annotation status, not sequence homology. A strongly supported sequence match should not be discarded solely because its functional annotation is incomplete.

In [ ]:
import hashlib
import json
import random
import re
import time

import pandas as pd
from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.SeqRecord import SeqRecord

STATE_DIR = OUTPUT_DIR / "_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)

SUCCESS_TSV = STATE_DIR / "successful_hits.tsv"
REJECT_TSV = STATE_DIR / "terminal_rejections.tsv"
ERROR_TSV = STATE_DIR / "transient_errors.tsv"
DONE_TSV = STATE_DIR / "done_taxa.tsv"
RAW_STATE_FASTA = STATE_DIR / "homologs_raw_incremental.faa"

SUCCESS_COLUMNS = [
    "label", "requested_taxid", "target_ncbi_name",
    "blast_accession", "accession", "organism", "actual_taxid", "uniprot",
    "bitscore", "evalue", "pident", "query_coverage",
    "alignment_length", "blast_title", "sequence_sha1"
]
REJECT_COLUMNS = ["label", "requested_taxid", "reason", "accession", "details"]
ERROR_COLUMNS = ["label", "requested_taxid", "stage", "error_type", "details"]

_last_blast_submission = 0.0

def append_tsv_row(path, row, columns):
    one = pd.DataFrame([[row.get(c, "") for c in columns]], columns=columns)
    one.to_csv(
        path,
        sep="\t",
        mode="a",
        header=not path.exists() or path.stat().st_size == 0,
        index=False
    )

def append_done(taxid, label):
    with open(DONE_TSV, "a") as handle:
        handle.write(f"{taxid}\t{label}\n")

def load_done_taxids():
    done = set()
    if DONE_TSV.exists():
        with open(DONE_TSV) as handle:
            for line in handle:
                line = line.strip()
                if line:
                    done.add(line.split("\t")[0])
    return done

def retry_call(function, stage="request"):
    delay = BACKOFF_START_SECONDS
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return function()
        except Exception as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                break
            print(
                f"  {stage} attempt {attempt} failed "
                f"({type(exc).__name__}); retrying..."
            )
            time.sleep(delay + random.random())
            delay *= 2
    raise last_error

def wait_for_blast_submission_slot():
    global _last_blast_submission
    elapsed = time.time() - _last_blast_submission
    remaining = MIN_SECONDS_BETWEEN_BLAST_SUBMISSIONS - elapsed
    if remaining > 0:
        time.sleep(remaining + random.random() * 0.25)
    _last_blast_submission = time.time()

def run_taxon_blast(query_sequence, taxid):
    wait_for_blast_submission_slot()
    organism_filter = f"txid{taxid}[ORGN]"
    handle = NCBIWWW.qblast(
        program=BLAST_PROGRAM,
        database=BLAST_DATABASE,
        sequence=query_sequence,
        entrez_query=organism_filter,
        expect=EVALUE,
        hitlist_size=HITLIST_SIZE,
        format_type="XML"
    )
    try:
        return NCBIXML.read(handle)
    finally:
        handle.close()

def alignment_stats(alignment, query_len):
    if not alignment.hsps:
        return None

    hsp = max(alignment.hsps, key=lambda x: x.bits)
    aligned_query_residues = abs(hsp.query_end - hsp.query_start) + 1
    coverage = aligned_query_residues / max(1, query_len)

    return {
        "bitscore": float(hsp.bits),
        "evalue": float(hsp.expect),
        "pident": 100.0 * hsp.identities / max(1, hsp.align_length),
        "query_coverage": coverage,
        "alignment_length": int(hsp.align_length),
        "blast_title": alignment.title or "",
    }

def choose_usable_alignment(blast_record, query_len):
    best_observed = None

    for alignment in blast_record.alignments:
        stats = alignment_stats(alignment, query_len)
        if stats is None:
            continue

        if best_observed is None:
            best_observed = (alignment, stats)

        if stats["query_coverage"] >= MIN_QUERY_COVERAGE:
            return alignment, stats, True

    if best_observed is not None:
        alignment, stats = best_observed
        return alignment, stats, False

    return None, None, False

def extract_source_taxid(gb_record):
    for feature in gb_record.features:
        if feature.type == "source":
            for xref in feature.qualifiers.get("db_xref", []):
                match = re.fullmatch(r"taxon:(\d+)", xref)
                if match:
                    return match.group(1)
    return ""

def extract_uniprot_ids(gb_record):
    hits = set()
    for feature in gb_record.features:
        for xref in feature.qualifiers.get("db_xref", []):
            match = re.search(
                r"UniProtKB(?:/[^:]+)?:([A-Z0-9]{6,10})",
                xref
            )
            if match:
                hits.add(match.group(1))
    return ";".join(sorted(hits))

def fetch_protein_genbank(accession):
    with Entrez.efetch(
        db="protein",
        id=accession,
        rettype="gb",
        retmode="text"
    ) as handle:
        return SeqIO.read(handle, "genbank")

def sequence_sha1(sequence):
    return hashlib.sha1(str(sequence).encode("utf-8")).hexdigest()

# Protect against accidentally resuming an old run after changing
# the query, target table, or important BLAST parameters.
RUN_SIGNATURE = STATE_DIR / "run_signature.json"
current_signature = {
    "query_sha1": hashlib.sha1(query_seq.encode("utf-8")).hexdigest(),
    "taxon_table_sha1": hashlib.sha1(TAXON_TARGETS.read_bytes()).hexdigest(),
    "blast_program": BLAST_PROGRAM,
    "blast_database": BLAST_DATABASE,
    "evalue": EVALUE,
    "hitlist_size": HITLIST_SIZE,
    "minimum_query_coverage": MIN_QUERY_COVERAGE,
}

if RUN_SIGNATURE.exists():
    saved_signature = json.loads(RUN_SIGNATURE.read_text())
    if saved_signature != current_signature:
        raise RuntimeError(
            "The saved BLAST state belongs to a different query, taxon table, "
            "or parameter set. Use the reset utility at the end of the notebook "
            "before starting this new analysis."
        )
else:
    RUN_SIGNATURE.write_text(json.dumps(current_signature, indent=2))

print("Helper functions are ready.")
print("State folder:", STATE_DIR)

## 8. Run the taxon-directed BLASTP searches

This is the long-running cell. Results are saved **after each completed taxon**.

If Colab disconnects or a network problem interrupts the run, reconnect to Drive and rerun the notebook from the top. TaxIDs listed in the state file are skipped automatically.

A temporary network failure is logged but **not** marked complete, so that taxon can be retried on the next run.

In [ ]:
done_taxids = load_done_taxids()
print(f"Previously completed targets: {len(done_taxids)}")

target_name_lookup = dict(
    zip(
        validated_targets["taxid"],
        validated_targets["ncbi_scientific_name"]
    )
)

for number, row in enumerate(targets.itertuples(index=False), start=1):
    label = row.label
    taxid = row.taxid

    if taxid in done_taxids:
        print(f"[{number:>2}/{len(targets)}] {label} (txid{taxid}) — already complete")
        continue

    print(f"[{number:>2}/{len(targets)}] {label} (txid{taxid})")

    # ----- 1. Remote taxon-restricted BLASTP -----
    try:
        blast_record = retry_call(
            lambda: run_taxon_blast(query_seq, taxid),
            stage="BLAST"
        )
    except Exception as exc:
        print("  BLAST failed:", type(exc).__name__, exc)
        append_tsv_row(
            ERROR_TSV,
            {
                "label": label,
                "requested_taxid": taxid,
                "stage": "blast",
                "error_type": type(exc).__name__,
                "details": str(exc),
            },
            ERROR_COLUMNS
        )
        continue

    alignment, stats, passed_coverage = choose_usable_alignment(
        blast_record, query_length
    )

    if alignment is None:
        print("  No BLAST hits.")
        append_tsv_row(
            REJECT_TSV,
            {
                "label": label,
                "requested_taxid": taxid,
                "reason": "no_hits",
                "accession": "",
                "details": "No BLASTP hits returned for this TaxID.",
            },
            REJECT_COLUMNS
        )
        append_done(taxid, label)
        done_taxids.add(taxid)
        continue

    blast_accession = alignment.accession

    if not passed_coverage:
        details = (
            f"Best observed candidate covered "
            f"{100 * stats['query_coverage']:.1f}% of the query; "
            f"minimum is {100 * MIN_QUERY_COVERAGE:.1f}%."
        )
        print("  Rejected:", details)
        append_tsv_row(
            REJECT_TSV,
            {
                "label": label,
                "requested_taxid": taxid,
                "reason": "low_query_coverage",
                "accession": blast_accession,
                "details": details,
            },
            REJECT_COLUMNS
        )
        append_done(taxid, label)
        done_taxids.add(taxid)
        continue

    # ----- 2. Fetch full protein record and metadata -----
    try:
        gb_record = retry_call(
            lambda: fetch_protein_genbank(blast_accession),
            stage="protein fetch"
        )
    except Exception as exc:
        print("  Protein fetch failed:", type(exc).__name__, exc)
        append_tsv_row(
            ERROR_TSV,
            {
                "label": label,
                "requested_taxid": taxid,
                "stage": "protein_fetch",
                "error_type": type(exc).__name__,
                "details": str(exc),
            },
            ERROR_COLUMNS
        )
        continue

    accession = gb_record.id
    organism = gb_record.annotations.get("organism", "")
    actual_taxid = extract_source_taxid(gb_record)
    uniprot = extract_uniprot_ids(gb_record)
    seq_hash = sequence_sha1(gb_record.seq)

    # ----- 3. Save sequence and metadata immediately -----
    output_record = SeqRecord(
        gb_record.seq,
        id=accession,
        description=(
            f"{label} | requested_txid:{taxid} | "
            f"organism:{organism}"
        )
    )
    with open(RAW_STATE_FASTA, "a") as handle:
        SeqIO.write([output_record], handle, "fasta")

    success_row = {
        "label": label,
        "requested_taxid": taxid,
        "target_ncbi_name": target_name_lookup.get(taxid, ""),
        "blast_accession": blast_accession,
        "accession": accession,
        "organism": organism,
        "actual_taxid": actual_taxid,
        "uniprot": uniprot,
        **stats,
        "sequence_sha1": seq_hash,
    }
    append_tsv_row(SUCCESS_TSV, success_row, SUCCESS_COLUMNS)

    append_done(taxid, label)
    done_taxids.add(taxid)

    print(
        f"  Accepted {accession}; "
        f"E={stats['evalue']:.2g}; "
        f"identity={stats['pident']:.1f}%; "
        f"query coverage={100 * stats['query_coverage']:.1f}%"
    )

print("\nBLAST acquisition pass finished.")
print(f"Completed targets: {len(done_taxids)} / {len(targets)}")

if ERROR_TSV.exists():
    print(
        "Transient errors were recorded. If completed targets are fewer than "
        "the total, rerun this cell to retry them."
    )

## 9. Build the final raw and curated homolog sets

`homologs_raw.faa` contains one accepted candidate for each successfully searched target.

The curated FASTA removes:
- repeated accessions; and
- exact duplicate amino-acid sequences that occur under different accessions.

This is **exact deduplication**, not sequence-identity clustering. Near-identical proteins remain in the curated set and can be examined later.

In [ ]:
import shutil

RAW_FASTA = OUTPUT_DIR / "homologs_raw.faa"
CURATED_FASTA = OUTPUT_DIR / "homologs_curated.faa"
METADATA_TSV = OUTPUT_DIR / "homolog_metadata.tsv"
PARAMETERS_TSV = OUTPUT_DIR / "acquisition_parameters.tsv"
REJECTED_HITS_TSV = OUTPUT_DIR / "rejected_hits.tsv"
ITOL_LABELS = OUTPUT_DIR / "iTOL_labels.tsv"
ITOL_POPUP = OUTPUT_DIR / "iTOL_popup.tsv"

if SUCCESS_TSV.exists():
    successes = pd.read_csv(SUCCESS_TSV, sep="\t", dtype=str).fillna("")
else:
    successes = pd.DataFrame(columns=SUCCESS_COLUMNS)

raw_records = (
    list(SeqIO.parse(RAW_STATE_FASTA, "fasta"))
    if RAW_STATE_FASTA.exists()
    else []
)

if len(successes) != len(raw_records):
    raise RuntimeError(
        "The incremental metadata and FASTA state files contain different "
        "numbers of records. Use the reset cell at the end of the notebook "
        "and rerun the acquisition step."
    )

SeqIO.write(raw_records, RAW_FASTA, "fasta")

curated_rows = []
curated_records = []
dedupe_rejections = []
seen_accessions = set()
seen_sequence_hashes = set()

for (_, meta), record in zip(successes.iterrows(), raw_records):
    accession = meta["accession"]
    seq_hash = meta["sequence_sha1"]

    if accession in seen_accessions:
        dedupe_rejections.append({
            "label": meta["label"],
            "requested_taxid": meta["requested_taxid"],
            "reason": "duplicate_accession",
            "accession": accession,
            "details": "The same protein accession was already selected for an earlier target.",
        })
        continue

    if seq_hash in seen_sequence_hashes:
        dedupe_rejections.append({
            "label": meta["label"],
            "requested_taxid": meta["requested_taxid"],
            "reason": "duplicate_sequence",
            "accession": accession,
            "details": "An identical amino-acid sequence was already retained under another accession.",
        })
        continue

    seen_accessions.add(accession)
    seen_sequence_hashes.add(seq_hash)
    curated_rows.append(meta.to_dict())
    curated_records.append(record)

curated_metadata = pd.DataFrame(curated_rows, columns=SUCCESS_COLUMNS)
curated_metadata.to_csv(METADATA_TSV, sep="\t", index=False)
SeqIO.write(curated_records, CURATED_FASTA, "fasta")

# Combine acquisition-stage rejections with deduplication rejections.
if REJECT_TSV.exists():
    acquisition_rejections = pd.read_csv(
        REJECT_TSV, sep="\t", dtype=str
    ).fillna("")
else:
    acquisition_rejections = pd.DataFrame(columns=REJECT_COLUMNS)

dedupe_df = pd.DataFrame(dedupe_rejections, columns=REJECT_COLUMNS)
all_rejections = pd.concat(
    [acquisition_rejections, dedupe_df],
    ignore_index=True
)
all_rejections.to_csv(REJECTED_HITS_TSV, sep="\t", index=False)

# Reproducibility record.
parameters = pd.DataFrame([
    ("query_file", str(QUERY_FASTA)),
    ("taxon_target_file", str(TAXON_TARGETS)),
    ("blast_program", BLAST_PROGRAM),
    ("blast_database", BLAST_DATABASE),
    ("evalue_threshold", EVALUE),
    ("hitlist_size", HITLIST_SIZE),
    ("minimum_query_coverage", MIN_QUERY_COVERAGE),
    ("number_of_taxon_targets", len(targets)),
    ("query_length_aa", query_length),
], columns=["parameter", "value"])
parameters.to_csv(PARAMETERS_TSV, sep="\t", index=False)

# Optional iTOL annotation files.
with open(ITOL_LABELS, "w") as handle:
    handle.write("LABELS\nSEPARATOR TAB\nDATA\n")
    for row in curated_metadata.itertuples(index=False):
        label_text = (
            f"{row.label} | {row.organism} | {row.accession}"
        ).replace("\t", " ")
        handle.write(f"{row.accession}\t{label_text}\n")

with open(ITOL_POPUP, "w") as handle:
    handle.write("POPUP_INFO\nSEPARATOR TAB\nDATA\n")
    for row in curated_metadata.itertuples(index=False):
        popup = (
            f"Requested target: {row.label}; "
            f"requested TaxID: {row.requested_taxid}; "
            f"organism: {row.organism}; "
            f"actual TaxID: {row.actual_taxid}; "
            f"BLAST E-value: {row.evalue}; "
            f"identity: {row.pident}%; "
            f"query coverage: {100 * float(row.query_coverage):.1f}%"
        ).replace("\t", " ")
        handle.write(f"{row.accession}\t{popup}\n")

print("Taxon targets:           ", len(targets))
print("Accepted raw candidates: ", len(raw_records))
print("Curated unique sequences:", len(curated_records))
print("Rejected/removed hits:   ", len(all_rejections))

## 10. Inspect the results before moving on

The table below is intentionally compact. Look for unusually weak E-values, low query coverage, repeated organisms, or other results that deserve manual inspection.

In [ ]:
if curated_metadata.empty:
    print("No curated homologs are available yet.")
else:
    view = curated_metadata[
        [
            "label", "organism", "accession",
            "evalue", "pident", "query_coverage"
        ]
    ].copy()

    view["pident"] = pd.to_numeric(view["pident"], errors="coerce").round(1)
    view["query_coverage"] = (
        100 * pd.to_numeric(view["query_coverage"], errors="coerce")
    ).round(1)

    view = view.rename(columns={
        "pident": "percent_identity",
        "query_coverage": "query_coverage_percent"
    })
    display(view)

if not all_rejections.empty:
    print("\nRejected or duplicate candidates:")
    display(all_rejections)

## 11. Interpretation

Answer these in complete, concise sentences.

1. Why does restricting separate BLASTP searches to selected TaxIDs produce a different homolog set than taking the top results from one unrestricted BLASTP search?
2. How many target taxa produced an accepted candidate, and how many unique sequences remained after exact deduplication?
3. Identify one candidate with relatively weak sequence support (for example, a higher E-value, lower percent identity, or lower query coverage). Why should it receive extra scrutiny?
4. Why does this notebook call the sequences **candidate homologs** rather than automatically calling them **orthologs**?
5. What limitation is introduced by choosing only one candidate sequence per taxonomic target?
6. Which file should be carried forward to multiple-sequence alignment, and why?

> **Carry forward:** use `homologs_curated.faa` as the standard input to the next alignment notebook.

## 12. Confirm the required files

Run this cell before you finish. All five required outputs should be present.

In [ ]:
required_outputs = [
    "homologs_raw.faa",
    "homologs_curated.faa",
    "homolog_metadata.tsv",
    "acquisition_parameters.tsv",
    "rejected_hits.tsv",
]

print("Required outputs:")
for name in required_outputs:
    path = OUTPUT_DIR / name
    print(" ✓" if path.exists() else " ✗", name)

extras = ["iTOL_labels.tsv", "iTOL_popup.tsv"]
print("\nOptional downstream files:")
for name in extras:
    path = OUTPUT_DIR / name
    print(" ✓" if path.exists() else " ✗", name)

## Instructor / troubleshooting utility — reset this notebook's BLAST state

Normally, **do not run a reset**. The saved state is what allows a long BLAST acquisition to resume after a Colab interruption.

If you deliberately want to discard the entire run and start again, change `RESET_OUTPUTS` to `True` and run the cell once. Then immediately change it back to `False`.

In [ ]:
import shutil

RESET_OUTPUTS = False

if RESET_OUTPUTS:
    if STATE_DIR.exists():
        shutil.rmtree(STATE_DIR)

    for name in [
        "homologs_raw.faa",
        "homologs_curated.faa",
        "homolog_metadata.tsv",
        "acquisition_parameters.tsv",
        "rejected_hits.tsv",
        "iTOL_labels.tsv",
        "iTOL_popup.tsv",
    ]:
        path = OUTPUT_DIR / name
        if path.exists():
            path.unlink()

    print("BLAST state and generated outputs were deleted.")
    print("Set RESET_OUTPUTS back to False before continuing.")
else:
    print("No files were changed. RESET_OUTPUTS is False.")